# Notebook 10 — Detuned / Dressed CZ Exploration

This notebook explores a **detuned, dressed-interaction regime** as a low-leakage alternative to resonant pulse sequences.

Instead of relying on strong resonant population transfer, we drive the system with:
- finite detuning $\Delta$,
- a controlled interaction strength $V$,
- an interaction time $t$,

and study whether the system can accumulate a useful **controlled phase** while keeping population closer to the computational basis.

The main questions are:

1. Can detuning reduce leakage?
2. Can the entangling phase approach $\pi$?
3. Is there a regime where phase-corrected gate quality improves over the resonant cases?


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'scipy', 'matplotlib']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import basis, qeye, tensor, sigmax, sesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]
basis_labels = ['|gg>', '|gr>', '|rg>', '|rr>']

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Hamiltonian and effective-unitary helpers

In [ ]:
def dressed_hamiltonian(omega: float, delta: float, V: float):
    drive = 0.5 * omega * (sx1 + sx2)
    detuning = -delta * (n1 + n2)
    interaction = V * n1 * n2
    return drive + detuning + interaction

def run_dressed_evolution(psi0, omega: float, delta: float, V: float,
                          t_gate: float, n_steps: int = 220):
    H = dressed_hamiltonian(omega=omega, delta=delta, V=V)
    times = np.linspace(0.0, t_gate, n_steps)
    result = sesolve(H, psi0, times)
    return result.states[-1]

def state_to_column(psi):
    return np.array([basis_state.overlap(psi) for basis_state in basis_states], dtype=complex)

def build_effective_unitary(omega: float, delta: float, V: float, t_gate: float):
    columns = []
    for psi0 in basis_states:
        psi_final = run_dressed_evolution(
            psi0,
            omega=omega,
            delta=delta,
            V=V,
            t_gate=t_gate,
        )
        columns.append(state_to_column(psi_final))
    return np.column_stack(columns)


## Gate metrics

In [ ]:
def process_fidelity(U_eff, U_target=U_cz):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def phase_corrected_target(U_eff):
    phi_ent = entangling_phase(U_eff)
    return np.diag([1, 1, 1, np.exp(1j * phi_ent)]).astype(complex)

def phase_corrected_fidelity(U_eff):
    return process_fidelity(U_eff, U_target=phase_corrected_target(U_eff))

def leakage_metric(U_eff):
    offdiag = U_eff.copy()
    np.fill_diagonal(offdiag, 0)
    return float(np.linalg.norm(offdiag))


## Compare resonant and detuned operating points

In [ ]:
omega = 1.0
V = 4.0

cases = [
    ('resonant', 0.0, 2*np.pi/omega),
    ('moderately detuned', 1.5, 4*np.pi/omega),
    ('strongly detuned', 3.0, 8*np.pi/omega),
]

for label, delta, t_gate in cases:
    U_eff = build_effective_unitary(omega=omega, delta=delta, V=V, t_gate=t_gate)
    print(label)
    print(f'  delta = {delta:.3f}, t_gate = {t_gate:.3f}')
    print(f'  raw fidelity:             {process_fidelity(U_eff):.6f}')
    print(f'  phase-corrected fidelity: {phase_corrected_fidelity(U_eff):.6f}')
    print(f'  entangling phase / pi:    {entangling_phase(U_eff)/np.pi:.6f}')
    print(f'  leakage:                  {leakage_metric(U_eff):.6f}')
    print()


## Effective-unitary magnitude for a detuned candidate

In [ ]:
delta_demo = 2.0
t_gate_demo = 6*np.pi/omega
U_demo = build_effective_unitary(omega=omega, delta=delta_demo, V=V, t_gate=t_gate_demo)

plt.figure(figsize=(6, 5))
im = plt.imshow(np.abs(U_demo), origin='lower', aspect='equal')
plt.xticks(range(4), basis_labels)
plt.yticks(range(4), basis_labels)
plt.xlabel('Input basis state')
plt.ylabel('Output basis amplitude')
plt.title(r'$|U_{eff}|$ for a detuned / dressed candidate')
plt.colorbar(im, label='magnitude')
plt.tight_layout()
plt.show()


## Scan gate time at fixed detuning

In [ ]:
delta = 2.0
t_vals = np.linspace(0.5*np.pi/omega, 10*np.pi/omega, 60)

raw_vals = []
pc_vals = []
phi_vals = []
leak_vals = []

for t_gate in t_vals:
    U_eff = build_effective_unitary(omega=omega, delta=delta, V=V, t_gate=t_gate)
    raw_vals.append(process_fidelity(U_eff))
    pc_vals.append(phase_corrected_fidelity(U_eff))
    phi_vals.append(entangling_phase(U_eff) / np.pi)
    leak_vals.append(leakage_metric(U_eff))


In [ ]:
plt.figure(figsize=(7.5, 4.8))
plt.plot(t_vals, raw_vals, label='raw fidelity')
plt.plot(t_vals, pc_vals, label='phase-corrected fidelity')
plt.xlabel('Gate time')
plt.ylabel('Fidelity')
plt.title(r'Detuned gate-time scan at fixed $\Delta$')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 4.8))
plt.plot(t_vals, phi_vals)
plt.axhline(1.0, linestyle='--', label='ideal +1')
plt.axhline(-1.0, linestyle='--', color='gray', label='ideal -1')
plt.xlabel('Gate time')
plt.ylabel(r'Entangling phase / $\pi$')
plt.title(r'Entangling phase for detuned evolution')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 4.8))
plt.plot(t_vals, leak_vals)
plt.xlabel('Gate time')
plt.ylabel('Leakage norm')
plt.title(r'Leakage for detuned evolution')
plt.tight_layout()
plt.show()


## 2D scan over detuning and gate time

In [ ]:
delta_vals = np.linspace(0.0, 4.0, 24)
t_gate_vals = np.linspace(0.5*np.pi/omega, 10*np.pi/omega, 30)

raw_map = np.zeros((len(delta_vals), len(t_gate_vals)))
pc_map = np.zeros((len(delta_vals), len(t_gate_vals)))
phi_map = np.zeros((len(delta_vals), len(t_gate_vals)))
leak_map = np.zeros((len(delta_vals), len(t_gate_vals)))

for i, delta_scan in enumerate(delta_vals):
    for j, t_gate_scan in enumerate(t_gate_vals):
        U_eff = build_effective_unitary(
            omega=omega,
            delta=delta_scan,
            V=V,
            t_gate=t_gate_scan,
        )
        raw_map[i, j] = process_fidelity(U_eff)
        pc_map[i, j] = phase_corrected_fidelity(U_eff)
        phi_map[i, j] = entangling_phase(U_eff) / np.pi
        leak_map[i, j] = leakage_metric(U_eff)


In [ ]:
plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(raw_map, origin='lower', aspect='auto', extent=[t_gate_vals[0], t_gate_vals[-1], delta_vals[0], delta_vals[-1]])
plt.contour(t_gate_vals, delta_vals, raw_map, colors='white', linewidths=0.4)
plt.xlabel('Gate time')
plt.ylabel(r'Detuning $\Delta$')
plt.title(r'Raw fidelity over $(t, \Delta)$')
plt.colorbar(im, label='raw fidelity')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(pc_map, origin='lower', aspect='auto', extent=[t_gate_vals[0], t_gate_vals[-1], delta_vals[0], delta_vals[-1]])
plt.contour(t_gate_vals, delta_vals, pc_map, colors='white', linewidths=0.4)
plt.xlabel('Gate time')
plt.ylabel(r'Detuning $\Delta$')
plt.title(r'Phase-corrected fidelity over $(t, \Delta)$')
plt.colorbar(im, label='phase-corrected fidelity')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(phi_map, origin='lower', aspect='auto', extent=[t_gate_vals[0], t_gate_vals[-1], delta_vals[0], delta_vals[-1]], vmin=-1, vmax=1)
plt.contour(t_gate_vals, delta_vals, phi_map, colors='white', linewidths=0.4)
plt.xlabel('Gate time')
plt.ylabel(r'Detuning $\Delta$')
plt.title(r'Entangling phase / $\pi$ over $(t, \Delta)$')
plt.colorbar(im, label=r'$\phi_{ent}/\pi$')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(leak_map, origin='lower', aspect='auto', extent=[t_gate_vals[0], t_gate_vals[-1], delta_vals[0], delta_vals[-1]])
plt.contour(t_gate_vals, delta_vals, leak_map, colors='white', linewidths=0.4)
plt.xlabel('Gate time')
plt.ylabel(r'Detuning $\Delta$')
plt.title(r'Leakage over $(t, \Delta)$')
plt.colorbar(im, label='leakage')
plt.tight_layout()
plt.show()


## Best detuned candidate in the $(t, \Delta)$ scan

In [ ]:
best_idx = np.unravel_index(np.argmax(pc_map), pc_map.shape)
best_delta = delta_vals[best_idx[0]]
best_tgate = t_gate_vals[best_idx[1]]

U_best = build_effective_unitary(omega=omega, delta=best_delta, V=V, t_gate=best_tgate)

print(f'Best detuned candidate: Delta = {best_delta:.4f}, t_gate = {best_tgate:.4f}')
print(f'  raw fidelity:             {process_fidelity(U_best):.6f}')
print(f'  phase-corrected fidelity: {phase_corrected_fidelity(U_best):.6f}')
print(f'  entangling phase / pi:    {entangling_phase(U_best)/np.pi:.6f}')
print(f'  leakage:                  {leakage_metric(U_best):.6f}')


## Interpretation

This notebook tests the idea that **detuning can trade population transfer for phase accumulation**.

The central question is whether a dressed-interaction regime can:
- keep leakage lower than resonant pulse sequences,
- still accumulate a useful entangling phase,
- improve phase-corrected gate quality.

If that works, it points toward the same design logic used in more realistic dressed or adiabatic Rydberg gates: **shape the trajectory so phase accumulates without strongly occupying the lossy manifold**.


## Optional next steps

Natural next upgrades are:

- make $\Omega(t)$ and $\Delta(t)$ explicitly time-dependent,
- use smooth ramps instead of square pulses,
- jointly optimize $(\Omega, \Delta, V, t)$,
- compare resonant, echo, and dressed protocols side by side in one final summary notebook.
